This lab cleans DSSI service (Direct Supply) searches from 2019 by cleaning up punctuation, numbers, and mispelling. I run into errors with bad decoding and odd things. 
it also compares frequent terms. 

In [1]:
%pip install pyspellchecker

Note: you may need to restart the kernel to use updated packages.


In [2]:
# In this cell: import spellcheck library and define your cleaning functions
import csv
import re
from spellchecker import SpellChecker
path = r"C:\Users\huelkes\Downloads\L2\searchTerms.csv"

def remove_punctuation(token):
    return re.sub(r"[^a-z0-9]", "", token)

def clean_numbers(token):
    if token.isdigit():
        return ""
    return token

def spell_check(token):
    return spell_dict.get(token, token)

def term_freq(term_list):
    freq = {}
    for t in term_list:
        if t in freq:
            freq[t] += 1
        else:
            freq[t] = 1
    return freq

def top_n(freq, n=10):
    return sorted(freq.items(), key=lambda x: x[1], reverse=True)[:n]

In [3]:
# In this cell: load the data, clean web spaces (%20) and split into
# individual list of terms. Display total number of terms in the data set.

phrases = []
with open(path, encoding = "utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if row:
            phrases.append(row[0])

tokens = []
for p in phrases:
    p = p.replace("%20", " ")
    for i in p.split():
        tokens.append(i.lower())


print("Total tokens: ", len(tokens))
        
    


Total tokens:  1469101


In [4]:
# In this cell: make first term frequency dictionary (step 4 in the lab document)
# Display the top 10 most common words in the dictionary
freq_raw = term_freq(tokens)
print("Distinct terms:", len(freq_raw))
for term, count in top_n(freq_raw):
    print(term, count)

Distinct terms: 61846
chicken 19208
cream 16045
cheese 13933
beef 13547
pie 11464
juice 11440
sauce 11274
pork 11098
potato 10137
green 9804


In [5]:
# In this cell: apply punctuation and number cleaning functions.
# Recreate the term frequency dictionary and display the updated top 10 list.
cleaned = []
for t in tokens:
    t = clean_numbers(remove_punctuation(t))
    if t != "":
        cleaned.append(t)

freq_clean = term_freq(cleaned)
print("Distinct terms:", len(freq_clean))
for term, count in top_n(freq_clean):
    print(term, count)

Distinct terms: 20202
chicken 19223
cream 16057
cheese 13957
beef 13566
pie 11475
juice 11452
sauce 11288
pork 11104
potato 10152
green 9804


In [6]:
%%time
# In this cell: apply spell checking function. Recreate the term frequency 
# dictionary and display the updated top 10 list.

spell = SpellChecker(distance=1)

spell_dict = {}
for w in spell.unknown(list(freq_clean.keys())):
    fix = spell.correction(w)
    if fix is not None:
        spell_dict[w] = fix

spelled = [spell_check(t) for t in cleaned]

freq_spelled = term_freq(spelled)
print("Distinct terms:", len(freq_spelled))
for term, count in top_n(freq_spelled):
    print(term, count)


print(len(freq_raw), len(freq_clean), len(freq_spelled))

print(list(spell_dict.items())[:50])   # sample of corrections

print([t for t in freq_raw if "%" in t][:30])          # leftover URL encodings
print([t for t in freq_raw if not t.isalnum()][:50])   # tokens with punctuation

Distinct terms: 17892
chicken 19230
cream 16057
cheese 14014
beef 13566
pie 11475
juice 11472
sauce 11296
pork 11108
potato 10649
green 9804
61846 20202 17892
[('sazon', 'salon'), ('frankes', 'frances'), ('ribbet', 'ribbed'), ('windo', 'wind'), ('tg', 'to'), ('fre', 'are'), ('vel', 'del'), ('blackberr', 'blackberry'), ('floure', 'flour'), ('razo', 'razor'), ('mm', 'my'), ('mascar', 'mascara'), ('fam', 'am'), ('pattys', 'patty'), ('pfm', 'pam'), ('pret', 'pet'), ('crispix', 'crispin'), ('yoo', 'you'), ('co2', 'cop'), ('guacomole', 'guacamole'), ('knigh', 'knight'), ('cho', 'who'), ('crispys', 'crispy'), ('throate', 'throat'), ('compand', 'company'), ('mudd', 'mud'), ('softners', 'softness'), ('1lb', 'alb'), ('quat', 'quit'), ('clea', 'clear'), ('blacke', 'black'), ('apricote', 'apricot'), ('zesta', 'zest'), ('ravil', 'rail'), ('mfld', 'mild'), ('pvb', 'pub'), ('az', 'a'), ('xspan', 'span'), ('suppos', 'suppose'), ('sbar', 'bar'), ('iw', 'i'), ('gu', 'go'), ('chapstick', 'chopstick'), ('

In [7]:
# In this cell: create required CSV.
out_path = r"C:\Users\huelkes\Downloads\L2\term_frequencies.csv"
with open(out_path, "w", encoding="utf-8") as f:
    for term, count in sorted(freq_spelled.items(), key=lambda x: x[1], reverse=True):
        f.write(f"{term}, {count}\n")

1) Compare the three sets of 10 most frequent terms that you created. Did your cleaning functions have a large impact on the most frequent words? Why might this be?
They were all very similar and it did not seem to have that much of a different. The count would go up but not really importnat. 

2) Compare the total number of distinct terms within each term frequency dictionary (i.e., how many keys were in each dictionary). Did your cleaning functions have a large impact on the total number of words in your data set? Elaborate.
There were 61,846 distinct terms in the raw data, 20,202 after removing punctuation and numbers (about a 67% drop), and 17,892 after spell checking (about another 11% drop).
i.e. cleaning had a pretty big impact

4) Do you believe that spell checking this data resulted in a more “clean” data set? Why or why not?
Sometimes, it showed to be helpful but also ended up spell checking from valid work to valid work and that can totally change the search for the user 

5) You probably noticed that there are punctuation marks and other “nonwords” in the data. Where do you think these originated from? Include few statements that hypothesize regarding the origins of some of these nonwords.
URL encoding that wasn't fully decoded? %22 is a quotation, %c3%a9 is e in puree with the special e. 

6) What is the big-O of the slowest cell, and what is the “wall time” when you benchmark it using %%time? If your list of search terms were 10x larger, how long would you estimate this cell to take? Note: You can assume the spell checker works in O(1) time.
o(n) and it was a CPU time of 13.3s and wall time of 13.5s. It was so much because it went through every n. Probably 10x  of whatever the current CPU time is (135 seconds here). 
